In [20]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

import random
self_seed = 0
g = torch.Generator().manual_seed(self_seed)

import math

In [2]:
block_sizes = [3,4,6]       # 上下文长度
embedding_dims = [2,5,10]   # 嵌入维度
hidden_dim = 100    # 隐藏层大小
batch_size = 32     # 批次数
lr_base = 0.1       # 初始学习率

In [ ]:
pinyins = open('data/names1/names1_pinyin_components.txt','r').read().splitlines()
pinyins_to_chars = open('data/names1/pinyin_components_to_hanzi_chars1.txt','r').read().splitlines()

tokens = [p.split() for p in pinyins]
token_list = sorted({'.'} | {t for token in tokens for t in token})
token_list_len = len(token_list)
stoi = {s:i for i,s in enumerate(token_list)}
itos = {i:s for i,s in enumerate(token_list)}

In [16]:
def build_dataset(data,block_size):
    X,Y = [],[]
    for t in data:
        # print(['.']*block_size + t + ['.'])
        # print(t + ['.'])

        str1 = ['.']*block_size + t + ['.']
        str2 = t + ['.']

        i=0
        for _,y in zip(str1,str2):
            # print(str1[i:i+block_size],y)
            # print([stoi[str1[j]] for j in range(i,i+block_size)],stoi[y])
            X.append([stoi[str1[j]] for j in range(i,i+block_size)])
            Y.append(stoi[y])
            i+=1


    X=torch.tensor(X)
    Y=torch.tensor(Y)
    #print(X.shape,Y.shape,X.dtype,Y.dtype)
    return X,Y

In [22]:
def get_lr(lr_base, i, max_steps, warmup_steps=10000, lr_min=1e-5):
    """
    余弦退火学习率
    warmup_steps: 预热步数，前1万步线性增加
    lr_min: 最低学习率，防止后期完全不动
    """
    # 1. 预热阶段：从 0 线性增加到 lr_base
    if i < warmup_steps:
        return lr_base * (i / warmup_steps)
    
    # 2. 余弦退火阶段：从 lr_base 平滑衰减到 lr_min
    # 注意要减去预热步数
    progress = (i - warmup_steps) / (max_steps - warmup_steps)
    # 使用 cosine 公式
    return lr_min + 0.5 * (lr_base - lr_min) * (1.0 + math.cos(math.pi * progress))

In [23]:
def init_model(embedding_dim,block_size):
    E = torch.randn((token_list_len, embedding_dim)        ,generator=g)
    W1 = torch.randn((block_size*embedding_dim, hidden_dim),generator=g) * (5/3)/((embedding_dim*block_size)**0.5)*0.4
    b1 = torch.randn(hidden_dim                            ,generator=g) * 0.2
    W2 = torch.randn((hidden_dim,token_list_len)           ,generator=g) * 0.01
    b2 = torch.randn(token_list_len                        ,generator=g) * 0
    return E,W1,b1,W2,b2

In [ ]:
all_lossi = []
for block_size in block_sizes:
    for embedding_dim in embedding_dims:
        # print(block_size,embedding_dim)
        # 训练集 80% -> 优化模型权重
        # 验证集 10% -> 调整超参数
        # 测试集 10% -> 测试模型泛化性和准确率
        random.seed(self_seed)
        random.shuffle(tokens)
        n1 = int(0.8*len(tokens))
        n2 = int(0.9*len(tokens))

        Xtr,Ytr   = build_dataset(tokens[:n1],block_size)
        Xdev,Ydev = build_dataset(tokens[n1:n2],block_size)
        Xte,Yte   = build_dataset(tokens[n2:],block_size)

        E,W1,b1,W2,b2 = init_model(embedding_dim,block_size)

        parameters=[E,W1,b1,W2,b2]
        for p in parameters:
            p.requires_grad = True

        # train
        max_steps = 200000
        lossi = []
        for i in range(max_steps):
            # forward pass
            batch_idxs = torch.randint(0,Xtr.shape[0],(batch_size,))
            emb = E[Xtr[batch_idxs]]
            hpreact = emb.view(-1,block_size*embedding_dim)@W1+b1 # h_practivate
            h = torch.tanh(hpreact)
            logits = h@W2+b2
            loss = F.cross_entropy(logits,Ytr[batch_idxs])

            #backward pass
            for p in parameters:
                p.grad = None
            loss.backward()

            # update
            lr = get_lr(lr_base,i,max_steps)
            for p in parameters:
                p.data += -lr*p.grad
            
            # track stats
            lossi.append(loss.item())

            # print
            if(i%50000==0):
                print(f'[{block_size,embedding_dim}]-> {i:6d}/{max_steps:6d}: loss={loss.item()}')
            #break
    
